# AI-MCN Team Execution Guide

## Phase Map
1. Phase A: Individual setup and data collection (ends when each member downloads their own ZIP).
2. Phase B: Team integration and analysis (starts only after all 4 member ZIPs are uploaded).

---

## Phase A — Individual Member Workflow

### A1) Create your Google Cloud project
1. Go to Google Cloud [Create Project](https://console.cloud.google.com/projectcreate).
2. Create one project per teammate (example: `ai-mcn-member1-youtube`).
3. Keep your Project ID for quota checks.

### A2) Enable YouTube Data API v3
1. Open [YouTube Data API v3 Library page](https://console.cloud.google.com/apis/library/youtube.googleapis.com).
2. Confirm your project is selected.
3. Click **Enable**.

### A3) Create API key
1. Open [Credentials](https://console.cloud.google.com/apis/credentials).
2. Click **Create credentials** -> **API key**. -> Select **PUBLIC** option
3. Copy the key and store it securely.

### A4) Restrict API key (required)
1. Edit the key in [Credentials](https://console.cloud.google.com/apis/credentials).
2. Set **API restrictions** to **YouTube Data API v3** only.
3. Save.
4. Reference: [Google API key best practices](https://cloud.google.com/docs/authentication/api-keys).

### A5) Add key to Colab Secrets
1. Open [Google Colab](https://colab.research.google.com/).
2. Open the notebook.
3. Click the **Secrets** panel 🔑 (key icon).
4. Add secret name: `YOUTUBE_API_KEY`.
5. Paste your key as value.
6. Turn on notebook access.

### A6) Update only your member config
1. In the config cell, set `MEMBER_NAME` exactly:
2. Alice: `member_1`
3. Sraddha: `member_2`
4. Alex: `member_3`
5. Nazim: `member_4`

### A7) Run notebook
1. Run all cells in order.
2. Monitor quota if needed at [Quota page](https://console.cloud.google.com/apis/api/youtube.googleapis.com/quotas).
3. Quota reference: [YouTube quota costs](https://developers.google.com/youtube/v3/determine_quota_cost).

### A8) Download and upload your outputs
1. Download `youtube_outputs_{member}.zip`.
2. Confirm ZIP contains 4 CSV files:
3. `youtube_master_full_{member}.csv`
4. `youtube_master_prd_slim_with_comments_{member}.csv`
5. `youtube_comments_raw_{member}.csv`
6. `youtube_videos_text_ready_{member}.csv`
7. Upload ZIP to shared team folder.

### A9) Phase A completion gate
1. Phase A is complete only when all 4 teammates uploaded their ZIP files.


## Troubleshooting
1. `YOUTUBE_API_KEY` error: re-check secret name and notebook access in Colab.
2. `quotaExceeded`: wait for reset or reduce collection scope.
3. Only one file downloaded: use ZIP download only.
4. Wrong member output: verify `MEMBER_NAME` before running.


In [ ]:
!pip -q install google-api-python-client pandas tqdm

In [ ]:
import os
import json
import random
import time
import logging
from datetime import datetime, timezone

import pandas as pd
from tqdm.auto import tqdm
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from google.colab import userdata

logging.getLogger("googleapiclient.http").setLevel(logging.ERROR)
logging.getLogger("googleapiclient.discovery_cache").setLevel(logging.ERROR)

API_KEY = userdata.get("YOUTUBE_API_KEY")
if not API_KEY:
    raise ValueError("Add YOUTUBE_API_KEY in Colab Secrets first.")

youtube = build("youtube", "v3", developerKey=API_KEY, cache_discovery=False)
print("YouTube client ready.")


YouTube client ready.


### ⭐️ Before Run below Cell, Update only your member config
1. In the config cell, set `MEMBER_NAME` exactly:
      2. Alice: `member_1`
      3. Sraddha: `member_2`
      4. Alex: `member_3`
      5. Nazim: `member_4`

In [ ]:
# =========================
# Team config
# =========================
MEMBER_NAME = "member_1"  # just change here to -> member_1 / member_2 / member_3 / member_4

QUERY_PACKS = {
    "member_1": [
        "beauty youtuber", "beauty influencer", "beauty vlog",
        "get ready with me beauty", "beauty favorites", "clean beauty", "everyday makeup look"
    ],
    "member_2": [
        "skincare routine", "skincare review", "dermatologist skincare",
        "acne skincare", "retinol routine", "sunscreen review", "sensitive skin skincare"
    ],
    "member_3": [
        "makeup tutorial", "drugstore makeup", "high end makeup review",
        "foundation review", "concealer review", "bridal makeup tutorial", "makeup declutter"
    ],
    "member_4": [
        "haircare routine", "curly hair routine", "fragrance review",
        "body care routine", "nail tutorial", "mens grooming", "beauty devices review"
    ]
}
QUERIES = QUERY_PACKS[MEMBER_NAME]

REGION_CODE = "US"
RELEVANCE_LANGUAGE = "en"


TARGET_CHANNELS_PER_QUERY = 40
MAX_PLAYLISTS_PER_CHANNEL = 12
MAX_ITEMS_PER_PLAYLIST = 30
MAX_UPLOAD_VIDEOS_PER_CHANNEL = 30

# Comment strategy
COMMENT_COLLECTION_SCOPE = "top_viewed"
TOP_VIEWED_COMMENT_VIDEO_LIMIT = 250
MAX_COMMENT_THREADS_PER_VIDEO = 20
COMMENT_ORDER = "relevance"
MAX_COMMENT_TEXT_CHARS = 5000


OUTPUT_DIR = "/content"

CHANNEL_PARTS_CANDIDATES = [
    "id,snippet,statistics,contentDetails,status,topicDetails,brandingSettings",
    "id,snippet,statistics,contentDetails,status,topicDetails",
    "id,snippet,statistics,contentDetails,status",
    "id,snippet,statistics,contentDetails"
]

VIDEO_PARTS_CANDIDATES = [
    "id,snippet,contentDetails,statistics,status,topicDetails,recordingDetails,liveStreamingDetails",
    "id,snippet,contentDetails,statistics,status,topicDetails",
    "id,snippet,contentDetails,statistics,status",
    "id,snippet,contentDetails,statistics"
]

PLAYLIST_PARTS = "id,snippet,contentDetails,status"
PLAYLIST_ITEM_PARTS = "id,snippet,contentDetails,status"
COMMENT_THREAD_PARTS = "id,snippet"

# =========================
# Built-in clean output columns
# =========================
MASTER_FULL_COLUMNS = [
    "_dedupe_key", "_endpoint", "_entity_type", "_part_used", "_member", "_query_label", "_collected_at",
    "id", "_resource_id", "_channel_id", "_video_id", "_playlist_id", "_comment_id", "_parent_comment_id",
    "snippet__publishedAt", "snippet__channelId", "snippet__channelTitle", "snippet__title", "snippet__description",
    "snippet__tags", "snippet__categoryId", "snippet__country",
    "snippet__playlistId", "snippet__position", "snippet__resourceId__videoId",
    "snippet__videoOwnerChannelId", "snippet__videoOwnerChannelTitle",
    "contentDetails__relatedPlaylists__uploads", "contentDetails__videoId", "contentDetails__videoPublishedAt",
    "contentDetails__duration", "contentDetails__definition", "contentDetails__caption", "contentDetails__licensedContent",
    "statistics__subscriberCount", "statistics__videoCount", "statistics__viewCount", "statistics__likeCount",
    "statistics__commentCount", "topicDetails__topicCategories",
    "brandingSettings__channel__keywords", "brandingSettings__channel__country",
    "status__privacyStatus", "status__license", "status__embeddable", "status__publicStatsViewable", "status__madeForKids",
    "comment_text", "comment_like_count", "comment_published_at", "comment_author", "comment_author_channel_id",
    "comment_count_collected", "comment_avg_like", "comment_text_concat"
]

MASTER_SLIM_COLUMNS = [
    "_dedupe_key", "_endpoint", "id", "_channel_id", "_video_id", "_playlist_id", "_comment_id",
    "snippet__publishedAt", "snippet__channelTitle", "snippet__title", "snippet__description", "snippet__tags", "snippet__categoryId",
    "contentDetails__duration",
    "statistics__subscriberCount", "statistics__videoCount", "statistics__viewCount", "statistics__likeCount", "statistics__commentCount",
    "topicDetails__topicCategories", "brandingSettings__channel__keywords", "status__madeForKids",
    "comment_text", "comment_like_count", "comment_published_at", "comment_author",
    "comment_count_collected", "comment_avg_like", "comment_text_concat"
]

COMMENTS_OUTPUT_COLUMNS = [
    "_dedupe_key", "_endpoint", "_member", "_query_label", "_collected_at",
    "_comment_id", "_parent_comment_id", "_video_id", "_channel_id",
    "comment_published_at", "comment_author", "comment_author_channel_id",
    "comment_like_count", "comment_text"
]

VIDEOS_OUTPUT_COLUMNS = [
    "_dedupe_key", "id", "_video_id", "_channel_id",
    "snippet__publishedAt", "snippet__channelId", "snippet__channelTitle",
    "snippet__title", "snippet__description", "snippet__tags", "snippet__categoryId",
    "contentDetails__duration", "contentDetails__definition", "contentDetails__caption",
    "statistics__viewCount", "statistics__likeCount", "statistics__commentCount",
    "topicDetails__topicCategories", "status__madeForKids",
    "comment_count_collected", "comment_avg_like", "comment_text_concat"
]


In [ ]:
def now_utc():
    return datetime.now(timezone.utc).isoformat()

def chunks(seq, n):
    for i in range(0, len(seq), n):
        yield seq[i:i+n]

def safe_get(d, path, default=None):
    cur = d
    for k in path:
        if not isinstance(cur, dict) or k not in cur:
            return default
        cur = cur[k]
    return cur

def series_from_candidates(df, candidates, default=""):
    for c in candidates:
        if c in df.columns:
            return df[c].astype("string")
    return pd.Series([default] * len(df), index=df.index, dtype="string")

def select_existing(df, cols):
    return df[[c for c in cols if c in df.columns]].copy()

def yt_execute(request, retries=5, base_sleep=1.0):
    last_err = None
    for attempt in range(retries):
        try:
            return request.execute()
        except HttpError as e:
            last_err = e
            status = getattr(e.resp, "status", None)
            msg = str(e).lower()
            retryable = status in (403, 429, 500, 503) or "quota" in msg or "rate" in msg
            if retryable and attempt < retries - 1:
                time.sleep(base_sleep * (2 ** attempt) + random.random())
                continue
            raise
    raise last_err

def execute_with_part_fallback(make_request, part_candidates):
    last_err = None
    for part in part_candidates:
        try:
            return yt_execute(make_request(part)), part
        except HttpError as e:
            last_err = e
            continue
    if last_err:
        raise last_err
    raise RuntimeError("No part candidates provided.")

def standardize_df(df, endpoint):
    if df.empty:
        return df

    out = df.copy()
    out["_resource_id"] = series_from_candidates(out, ["id", "id__videoId", "id__channelId", "id__playlistId"])
    out["_channel_id"] = series_from_candidates(out, [
        "snippet__channelId",
        "id__channelId",
        "snippet__resourceId__channelId",
        "snippet__topLevelComment__snippet__channelId",
        "contentDetails__channelId"
    ])
    out["_video_id"] = series_from_candidates(out, [
        "contentDetails__videoId",
        "id__videoId",
        "snippet__resourceId__videoId",
        "snippet__videoId",
        "snippet__topLevelComment__snippet__videoId"
    ])
    out["_playlist_id"] = series_from_candidates(out, [
        "snippet__playlistId",
        "id__playlistId",
        "contentDetails__relatedPlaylists__uploads"
    ])

    out["_comment_id"] = pd.Series([None] * len(out), index=out.index, dtype="string")
    if endpoint == "commentThreads.list" and "id" in out.columns:
        out["_comment_id"] = out["id"].astype("string")

    out["_parent_comment_id"] = series_from_candidates(out, ["snippet__parentId"])

    if endpoint == "channels.list" and "id" in out.columns:
        out["_channel_id"] = out["id"].astype("string")
        out["_resource_id"] = out["id"].astype("string")
    if endpoint == "videos.list" and "id" in out.columns:
        out["_video_id"] = out["id"].astype("string")
        out["_resource_id"] = out["id"].astype("string")
    if endpoint == "playlists.list" and "id" in out.columns:
        out["_playlist_id"] = out["id"].astype("string")
        out["_resource_id"] = out["id"].astype("string")

    return out

def build_endpoint_df(items, endpoint, entity_type, part_used, member_name, query_label):
    if not items:
        return pd.DataFrame()

    df = pd.json_normalize(items, sep="__")
    df["_endpoint"] = endpoint
    df["_entity_type"] = entity_type
    df["_part_used"] = part_used
    df["_member"] = member_name
    df["_query_label"] = query_label
    df["_collected_at"] = now_utc()
    df["_raw_json"] = [json.dumps(item, ensure_ascii=False) for item in items]
    return standardize_df(df, endpoint)

def dedupe_master(df):
    if df.empty:
        return df

    out = df.copy()
    for c in ["_endpoint", "_resource_id", "_channel_id", "_video_id", "_playlist_id", "_comment_id", "_parent_comment_id"]:
        if c not in out.columns:
            out[c] = ""

    out["_dedupe_key"] = (
        out["_endpoint"].astype(str).fillna("") + "|" +
        out["_resource_id"].astype(str).fillna("") + "|" +
        out["_channel_id"].astype(str).fillna("") + "|" +
        out["_video_id"].astype(str).fillna("") + "|" +
        out["_playlist_id"].astype(str).fillna("") + "|" +
        out["_comment_id"].astype(str).fillna("") + "|" +
        out["_parent_comment_id"].astype(str).fillna("")
    )
    out = out.drop_duplicates(subset=["_dedupe_key", "_raw_json"], keep="first").reset_index(drop=True)
    return out

def concat_limited_texts(texts, max_chars=5000):
    out = []
    total = 0
    for t in texts:
        if pd.isna(t):
            continue
        s = " ".join(str(t).split())
        if not s:
            continue
        add_len = len(s) + (1 if out else 0)
        if total + add_len > max_chars:
            break
        out.append(s)
        total += add_len
    return " ".join(out)


In [ ]:
def search_channel_ids(youtube, query, target_channels):
    items, channel_ids, seen = [], [], set()
    page_token = None

    while len(channel_ids) < target_channels:
        req = youtube.search().list(
            part="snippet",
            q=query,
            type="channel",
            maxResults=min(50, target_channels - len(channel_ids)),
            regionCode=REGION_CODE,
            relevanceLanguage=RELEVANCE_LANGUAGE,
            pageToken=page_token
        )
        res = yt_execute(req)
        batch = res.get("items", [])
        if not batch:
            break

        items.extend(batch)
        for it in batch:
            cid = safe_get(it, ["id", "channelId"]) or safe_get(it, ["snippet", "channelId"])
            if cid and cid not in seen:
                seen.add(cid)
                channel_ids.append(cid)
                if len(channel_ids) >= target_channels:
                    break

        page_token = res.get("nextPageToken")
        if not page_token:
            break

    return items, channel_ids

def channels_list(youtube, channel_ids):
    all_items, used_parts = [], set()
    for batch in tqdm(list(chunks(channel_ids, 50)), desc="channels.list"):
        res, used = execute_with_part_fallback(
            lambda p: youtube.channels().list(part=p, id=",".join(batch), maxResults=50),
            CHANNEL_PARTS_CANDIDATES
        )
        used_parts.add(used)
        all_items.extend(res.get("items", []))
    return all_items, " | ".join(sorted(used_parts))

def playlists_list(youtube, channel_id, max_items):
    items, page_token = [], None
    while len(items) < max_items:
        req = youtube.playlists().list(
            part=PLAYLIST_PARTS,
            channelId=channel_id,
            maxResults=min(50, max_items - len(items)),
            pageToken=page_token
        )
        res = yt_execute(req)
        batch = res.get("items", [])
        if not batch:
            break
        items.extend(batch)
        page_token = res.get("nextPageToken")
        if not page_token:
            break
    return items

def playlist_items_list(youtube, playlist_id, max_items):
    items, page_token = [], None
    while len(items) < max_items:
        req = youtube.playlistItems().list(
            part=PLAYLIST_ITEM_PARTS,
            playlistId=playlist_id,
            maxResults=min(50, max_items - len(items)),
            pageToken=page_token
        )
        res = yt_execute(req)
        batch = res.get("items", [])
        if not batch:
            break
        items.extend(batch)
        page_token = res.get("nextPageToken")
        if not page_token:
            break
    return items

def videos_list(youtube, video_ids):
    all_items, used_parts = [], set()
    for batch in tqdm(list(chunks(video_ids, 50)), desc="videos.list"):
        res, used = execute_with_part_fallback(
            lambda p: youtube.videos().list(part=p, id=",".join(batch)),
            VIDEO_PARTS_CANDIDATES
        )
        used_parts.add(used)
        all_items.extend(res.get("items", []))
    return all_items, " | ".join(sorted(used_parts))

def comment_threads_list(youtube, video_id, max_items):
    items, page_token = [], None
    while len(items) < max_items:
        req = youtube.commentThreads().list(
            part=COMMENT_THREAD_PARTS,
            videoId=video_id,
            maxResults=min(100, max_items - len(items)),
            textFormat="plainText",
            order=COMMENT_ORDER,
            pageToken=page_token
        )
        try:
            res = yt_execute(req, retries=3)
        except HttpError as e:
            status = getattr(e.resp, "status", None)
            msg = str(e).lower()
            if status in (403, 404) or "commentsdisabled" in msg:
                break
            raise

        batch = res.get("items", [])
        if not batch:
            break
        items.extend(batch)
        page_token = res.get("nextPageToken")
        if not page_token:
            break
    return items


In [ ]:
def run_collection(youtube):
    query_label = ", ".join(QUERIES)
    all_dfs = []

    search_items_all = []
    channel_ids_all = []

    for q in QUERIES:
        s_items, c_ids = search_channel_ids(youtube, q, TARGET_CHANNELS_PER_QUERY)
        search_items_all.extend(s_items)
        channel_ids_all.extend(c_ids)

    channel_ids_all = list(dict.fromkeys(channel_ids_all))

    search_df = build_endpoint_df(search_items_all, "search.list", "search_result", "snippet", MEMBER_NAME, query_label)
    if not search_df.empty:
        all_dfs.append(search_df)

    channels_items, channels_parts_used = channels_list(youtube, channel_ids_all)
    channels_df = build_endpoint_df(channels_items, "channels.list", "channel", channels_parts_used, MEMBER_NAME, query_label)
    if not channels_df.empty:
        all_dfs.append(channels_df)

    uploads_map = {}
    if not channels_df.empty and "contentDetails__relatedPlaylists__uploads" in channels_df.columns and "id" in channels_df.columns:
        temp = channels_df[["id", "contentDetails__relatedPlaylists__uploads"]].dropna().drop_duplicates()
        uploads_map = dict(zip(temp["id"].astype(str), temp["contentDetails__relatedPlaylists__uploads"].astype(str)))

    all_playlist_ids = set(uploads_map.values())
    all_video_ids = set()

    for cid in tqdm(channel_ids_all, desc="playlists.list"):
        try:
            p_items = playlists_list(youtube, cid, MAX_PLAYLISTS_PER_CHANNEL)
            if p_items:
                p_df = build_endpoint_df(p_items, "playlists.list", "playlist", PLAYLIST_PARTS, MEMBER_NAME, query_label)
                all_dfs.append(p_df)
                if "id" in p_df.columns:
                    all_playlist_ids.update(p_df["id"].dropna().astype(str).tolist())
        except HttpError:
            pass

    for pid in tqdm(list(all_playlist_ids), desc="playlistItems.list"):
        try:
            pi_items = playlist_items_list(youtube, pid, MAX_ITEMS_PER_PLAYLIST)
            if not pi_items:
                continue
            pi_df = build_endpoint_df(pi_items, "playlistItems.list", "playlist_item", PLAYLIST_ITEM_PARTS, MEMBER_NAME, query_label)
            all_dfs.append(pi_df)
            if "contentDetails__videoId" in pi_df.columns:
                all_video_ids.update(pi_df["contentDetails__videoId"].dropna().astype(str).tolist())
        except HttpError:
            pass

    for cid, uploads_pid in tqdm(uploads_map.items(), desc="uploads playlistItems.list"):
        try:
            up_items = playlist_items_list(youtube, uploads_pid, MAX_UPLOAD_VIDEOS_PER_CHANNEL)
            if not up_items:
                continue
            up_df = build_endpoint_df(up_items, "playlistItems.list", "upload_playlist_item", PLAYLIST_ITEM_PARTS, MEMBER_NAME, query_label)
            all_dfs.append(up_df)
            if "contentDetails__videoId" in up_df.columns:
                all_video_ids.update(up_df["contentDetails__videoId"].dropna().astype(str).tolist())
        except HttpError:
            pass

    all_video_ids = list(dict.fromkeys([v for v in all_video_ids if isinstance(v, str) and v]))

    if all_video_ids:
        v_items, v_parts_used = videos_list(youtube, all_video_ids)
        v_df = build_endpoint_df(v_items, "videos.list", "video", v_parts_used, MEMBER_NAME, query_label)
        if not v_df.empty:
            all_dfs.append(v_df)

        target_video_ids = all_video_ids
        if COMMENT_COLLECTION_SCOPE == "top_viewed" and not v_df.empty and "statistics__viewCount" in v_df.columns:
            tmp = v_df[["id", "statistics__viewCount"]].copy()
            tmp["statistics__viewCount"] = pd.to_numeric(tmp["statistics__viewCount"], errors="coerce").fillna(0)
            target_video_ids = (
                tmp.sort_values("statistics__viewCount", ascending=False)["id"]
                .dropna().astype(str).head(TOP_VIEWED_COMMENT_VIDEO_LIMIT).tolist()
            )

        for vid in tqdm(target_video_ids, desc="commentThreads.list"):
            ct_items = comment_threads_list(youtube, vid, MAX_COMMENT_THREADS_PER_VIDEO)
            if not ct_items:
                continue

            ct_df = build_endpoint_df(ct_items, "commentThreads.list", "comment_thread", COMMENT_THREAD_PARTS, MEMBER_NAME, query_label)

            if "_video_id" not in ct_df.columns:
                ct_df["_video_id"] = vid
            else:
                ct_df["_video_id"] = ct_df["_video_id"].fillna(vid)

            ct_df["comment_text"] = [safe_get(t, ["snippet", "topLevelComment", "snippet", "textDisplay"], "") for t in ct_items]
            ct_df["comment_like_count"] = [safe_get(t, ["snippet", "topLevelComment", "snippet", "likeCount"], 0) for t in ct_items]
            ct_df["comment_published_at"] = [safe_get(t, ["snippet", "topLevelComment", "snippet", "publishedAt"], None) for t in ct_items]
            ct_df["comment_author"] = [safe_get(t, ["snippet", "topLevelComment", "snippet", "authorDisplayName"], None) for t in ct_items]
            ct_df["comment_author_channel_id"] = [safe_get(t, ["snippet", "topLevelComment", "snippet", "authorChannelId", "value"], None) for t in ct_items]
            all_dfs.append(ct_df)

    if not all_dfs:
        return pd.DataFrame()

    master_df = pd.concat(all_dfs, ignore_index=True, sort=False)
    master_df = dedupe_master(master_df)
    return master_df


In [ ]:
master_df = run_collection(youtube)

if master_df.empty:
    raise RuntimeError("No data collected. Check API key/quota/query settings.")

comments_df = master_df[master_df["_endpoint"] == "commentThreads.list"].copy() if "_endpoint" in master_df.columns else pd.DataFrame()
videos_df = master_df[master_df["_endpoint"] == "videos.list"].copy() if "_endpoint" in master_df.columns else pd.DataFrame()

comment_summary = pd.DataFrame(columns=["video_id", "comment_count_collected", "comment_avg_like", "comment_text_concat"])

if not comments_df.empty:
    if "_comment_id" not in comments_df.columns and "id" in comments_df.columns:
        comments_df["_comment_id"] = comments_df["id"].astype("string")
    elif "id" in comments_df.columns:
        comments_df["_comment_id"] = comments_df["_comment_id"].fillna(comments_df["id"].astype("string"))

    if "_video_id" in comments_df.columns and "snippet__videoId" in comments_df.columns:
        comments_df["_video_id"] = comments_df["_video_id"].fillna(comments_df["snippet__videoId"].astype("string"))

    comments_df["comment_like_count"] = pd.to_numeric(comments_df.get("comment_like_count", 0), errors="coerce").fillna(0)

    comment_summary = (
        comments_df.groupby("_video_id", dropna=True, as_index=False)
        .agg(
            comment_count_collected=("_comment_id", "nunique"),
            comment_avg_like=("comment_like_count", "mean"),
            comment_text_concat=("comment_text", lambda s: concat_limited_texts(s, MAX_COMMENT_TEXT_CHARS)),
        )
        .rename(columns={"_video_id": "video_id"})
    )

if not videos_df.empty and not comment_summary.empty and "id" in videos_df.columns:
    videos_df = videos_df.merge(comment_summary, left_on="id", right_on="video_id", how="left")

if not comment_summary.empty and "id" in master_df.columns and "_endpoint" in master_df.columns:
    smap = comment_summary.set_index("video_id")
    video_mask = master_df["_endpoint"] == "videos.list"
    for col in ["comment_count_collected", "comment_avg_like", "comment_text_concat"]:
        if col not in master_df.columns:
            master_df[col] = None
        master_df.loc[video_mask, col] = master_df.loc[video_mask, "id"].map(smap[col])

master_full_df = select_existing(master_df, MASTER_FULL_COLUMNS)
master_slim_df = select_existing(master_df, MASTER_SLIM_COLUMNS)
comments_out_df = select_existing(comments_df, COMMENTS_OUTPUT_COLUMNS)
videos_out_df = select_existing(videos_df, VIDEOS_OUTPUT_COLUMNS)

if "_dedupe_key" in master_full_df.columns:
    master_full_df = master_full_df.drop_duplicates(subset=["_dedupe_key"], keep="first")
else:
    master_full_df = master_full_df.drop_duplicates()

if "_dedupe_key" in master_slim_df.columns:
    master_slim_df = master_slim_df.drop_duplicates(subset=["_dedupe_key"], keep="first")
else:
    master_slim_df = master_slim_df.drop_duplicates()

if "_comment_id" in comments_out_df.columns:
    comments_out_df = comments_out_df.drop_duplicates(subset=["_comment_id"], keep="first")
else:
    comments_out_df = comments_out_df.drop_duplicates()

if "id" in videos_out_df.columns:
    videos_out_df = videos_out_df.drop_duplicates(subset=["id"], keep="first")
else:
    videos_out_df = videos_out_df.drop_duplicates()

os.makedirs(OUTPUT_DIR, exist_ok=True)

FULL_PATH = f"{OUTPUT_DIR}/youtube_master_full_{MEMBER_NAME}.csv"
SLIM_PATH = f"{OUTPUT_DIR}/youtube_master_prd_slim_with_comments_{MEMBER_NAME}.csv"
COMMENTS_PATH = f"{OUTPUT_DIR}/youtube_comments_raw_{MEMBER_NAME}.csv"
VIDEOS_PATH = f"{OUTPUT_DIR}/youtube_videos_text_ready_{MEMBER_NAME}.csv"
ZIP_PATH = f"{OUTPUT_DIR}/youtube_outputs_{MEMBER_NAME}.zip"

master_full_df.to_csv(FULL_PATH, index=False, encoding="utf-8")
master_slim_df.to_csv(SLIM_PATH, index=False, encoding="utf-8")
comments_out_df.to_csv(COMMENTS_PATH, index=False, encoding="utf-8")
videos_out_df.to_csv(VIDEOS_PATH, index=False, encoding="utf-8")

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in [FULL_PATH, SLIM_PATH, COMMENTS_PATH, VIDEOS_PATH]:
        zf.write(p, arcname=os.path.basename(p))

print("Saved:")
print(FULL_PATH, master_full_df.shape)
print(SLIM_PATH, master_slim_df.shape)
print(COMMENTS_PATH, comments_out_df.shape)
print(VIDEOS_PATH, videos_out_df.shape)
print("ZIP:", ZIP_PATH)
print("COMMENT_COLLECTION_SCOPE:", COMMENT_COLLECTION_SCOPE)
print("MAX_COMMENT_THREADS_PER_VIDEO:", MAX_COMMENT_THREADS_PER_VIDEO)

# ===== CSV summary print =====
SHOW_ALL_COLUMNS = True
MAX_COL_PREVIEW = 80

def print_df_summary(name, path, df):
    rows, cols = df.shape
    size_mb = os.path.getsize(path) / (1024 * 1024) if os.path.exists(path) else 0
    print(f"\n[{name}]")
    print(f"- path: {path}")
    print(f"- rows: {rows:,}")
    print(f"- cols: {cols}")
    print(f"- file size: {size_mb:.2f} MB")
    print("- columns:")

    col_list = list(df.columns)
    if not SHOW_ALL_COLUMNS and len(col_list) > MAX_COL_PREVIEW:
        col_list = col_list[:MAX_COL_PREVIEW]

    for i, c in enumerate(col_list, 1):
        print(f"  {i:03d}. {c}")

    if not SHOW_ALL_COLUMNS and len(df.columns) > MAX_COL_PREVIEW:
        print(f"  ... ({len(df.columns) - MAX_COL_PREVIEW} more columns)")

print_df_summary("MASTER_FULL", FULL_PATH, master_full_df)
print_df_summary("MASTER_SLIM", SLIM_PATH, master_slim_df)
print_df_summary("COMMENTS_RAW", COMMENTS_PATH, comments_out_df)
print_df_summary("VIDEOS_TEXT_READY", VIDEOS_PATH, videos_out_df)


channels.list:   0%|          | 0/6 [00:00<?, ?it/s]

playlists.list:   0%|          | 0/271 [00:00<?, ?it/s]

playlistItems.list:   0%|          | 0/1251 [00:00<?, ?it/s]

uploads playlistItems.list:   0%|          | 0/271 [00:00<?, ?it/s]

videos.list:   0%|          | 0/328 [00:00<?, ?it/s]

commentThreads.list:   0%|          | 0/250 [00:00<?, ?it/s]

Saved:
/content/youtube_master_full_member_1.csv (38206, 55)
/content/youtube_master_prd_slim_with_comments_member_1.csv (38206, 29)
/content/youtube_comments_raw_member_1.csv (2540, 14)
/content/youtube_videos_text_ready_member_1.csv (15671, 22)
ZIP: /content/youtube_outputs_member_1.zip
COMMENT_COLLECTION_SCOPE: top_viewed
MAX_COMMENT_THREADS_PER_VIDEO: 20

[MASTER_FULL]
- path: /content/youtube_master_full_member_1.csv
- rows: 38,206
- cols: 55
- file size: 72.92 MB
- columns:
  001. _dedupe_key
  002. _endpoint
  003. _entity_type
  004. _part_used
  005. _member
  006. _query_label
  007. _collected_at
  008. id
  009. _resource_id
  010. _channel_id
  011. _video_id
  012. _playlist_id
  013. _comment_id
  014. _parent_comment_id
  015. snippet__publishedAt
  016. snippet__channelId
  017. snippet__channelTitle
  018. snippet__title
  019. snippet__description
  020. snippet__tags
  021. snippet__categoryId
  022. snippet__country
  023. snippet__playlistId
  024. snippet__positi

In [ ]:
import zipfile
from google.colab import files
files.download(ZIP_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ===== Channel coverage summary =====
def non_empty_nunique(df, col):
    if col not in df.columns:
        return 0
    s = df[col].dropna().astype("string").str.strip()
    s = s[s != ""]
    return int(s.nunique())

total_channels = non_empty_nunique(master_full_df, "_channel_id")
video_channels = non_empty_nunique(videos_out_df, "_channel_id")
comment_channels = non_empty_nunique(comments_out_df, "_channel_id")

print("\n[CHANNEL COVERAGE]")
print(f"- total unique channels (_channel_id, master_full): {total_channels:,}")
print(f"- channels in videos_text_ready: {video_channels:,}")
print(f"- channels in comments_raw: {comment_channels:,}")



[CHANNEL COVERAGE]
- total unique channels (_channel_id, master_full): 682
- channels in videos_text_ready: 679
- channels in comments_raw: 68


In [ ]:
import numpy as np

def _non_empty_series(df, col):
    if col not in df.columns:
        return pd.Series([], dtype="string")
    s = df[col].dropna().astype("string").str.strip()
    return s[s != ""]

def _key_check(df, col):
    s = _non_empty_series(df, col)
    filled = len(s)
    dup = int(s.duplicated().sum())
    return filled, dup

print("\n[EXTRA SUMMARY]")

# 1) Endpoint distribution
if "_endpoint" in master_full_df.columns:
    print("\n- Endpoint distribution (master_full)")
    ep = master_full_df["_endpoint"].value_counts(dropna=False)
    total = len(master_full_df)
    for k, v in ep.items():
        print(f"  {k}: {v:,} ({v/total*100:.2f}%)")

# 2) Key integrity
print("\n- Key integrity")
for name, df, key in [
    ("master_full", master_full_df, "_dedupe_key"),
    ("master_slim", master_slim_df, "_dedupe_key"),
    ("videos", videos_out_df, "id"),
    ("comments", comments_out_df, "_comment_id"),
]:
    filled, dup = _key_check(df, key)
    print(f"  {name}.{key}: filled={filled:,}, dup={dup:,}")

# 3) Join coverage comments -> videos
video_ids = set(_non_empty_series(videos_out_df, "id").tolist())
comment_video_ids = set(_non_empty_series(comments_out_df, "_video_id").tolist())
matched = len(video_ids & comment_video_ids)
match_rate = (matched / len(comment_video_ids) * 100) if comment_video_ids else 0
print("\n- Join coverage (comments._video_id -> videos.id)")
print(f"  comment_video_ids: {len(comment_video_ids):,}")
print(f"  matched_video_ids: {matched:,}")
print(f"  match_rate: {match_rate:.2f}%")

# 4) Videos with comments ratio
videos_with_comments_ratio = (matched / len(video_ids) * 100) if video_ids else 0
print("\n- Videos with at least 1 collected comment")
print(f"  {matched:,} / {len(video_ids):,} ({videos_with_comments_ratio:.2f}%)")

# 5) Comment-per-video distribution
if "_video_id" in comments_out_df.columns and "_comment_id" in comments_out_df.columns:
    per_video = comments_out_df.groupby("_video_id")["_comment_id"].nunique()
    if len(per_video) > 0:
        print("\n- Comments per video distribution")
        print(f"  min={per_video.min()}, median={per_video.median():.1f}, p90={per_video.quantile(0.9):.1f}, max={per_video.max()}")

# 6) Top channels by video count
if "_channel_id" in videos_out_df.columns:
    top_ch = videos_out_df.groupby("_channel_id").size().sort_values(ascending=False).head(10)
    print("\n- Top 10 channels by video rows")
    for ch, cnt in top_ch.items():
        print(f"  {ch}: {cnt:,}")



[EXTRA SUMMARY]

- Endpoint distribution (master_full)
  playlistItems.list: 18,473 (48.35%)
  videos.list: 15,671 (41.02%)
  commentThreads.list: 2,540 (6.65%)
  playlists.list: 980 (2.57%)
  channels.list: 271 (0.71%)
  search.list: 271 (0.71%)

- Key integrity
  master_full._dedupe_key: filled=38,206, dup=0
  master_slim._dedupe_key: filled=38,206, dup=0
  videos.id: filled=15,671, dup=0
  comments._comment_id: filled=2,540, dup=0

- Join coverage (comments._video_id -> videos.id)
  comment_video_ids: 127
  matched_video_ids: 127
  match_rate: 100.00%

- Videos with at least 1 collected comment
  127 / 15,671 (0.81%)

- Comments per video distribution
  min=20, median=20.0, p90=20.0, max=20

- Top 10 channels by video rows
  UCs46s_D716dVmVpFSKWz23g: 252
  UCtNdVINwfYFTQEEZgMiQ8FA: 231
  UCPHuYCxf9QJArw-4ZXXC7Ig: 227
  UCHwY7H-o_S0lvtxzXTCr5gQ: 222
  UCCydBRF6RjrT58FV6KO-wwg: 217
  UC8oF7jmQsKbMVrCv5LrP32w: 211
  UCCz1ShCbAb134UtvWRQKBog: 210
  UCuflfWUNugwPkZOQUEyAowQ: 198
  UCgVx

# 2) Merge 4 teammate CSVs into one (After getting all each 4 memeber 's CSV files)


_channel_id is the unique identifier for each YouTube channel, so use _channel_id to identify channels.
For our team workflow, join the CSV files using _channel_id for channel-level integration.

import pandas as pd

slim_paths = [
    "/content/youtube_master_prd_slim_with_comments_member_1.csv",
    "/content/youtube_master_prd_slim_with_comments_member_2.csv",
    "/content/youtube_master_prd_slim_with_comments_member_3.csv",
    "/content/youtube_master_prd_slim_with_comments_member_4.csv",
]

slim_frames = [pd.read_csv(p, low_memory=False, dtype={"_channel_id":"string","_video_id":"string"}) for p in slim_paths]
slim_team = pd.concat(slim_frames, ignore_index=True, sort=False)

if "_dedupe_key" in slim_team.columns:
    slim_team = slim_team.drop_duplicates(subset=["_dedupe_key"], keep="first").reset_index(drop=True)

slim_team_out = "/content/youtube_team_prd_slim_with_comments.csv"
slim_team.to_csv(slim_team_out, index=False, encoding="utf-8")

print("Saved:", slim_team_out, slim_team.shape)


comments_paths = [
    "/content/youtube_comments_raw_member_1.csv",
    "/content/youtube_comments_raw_member_2.csv",
    "/content/youtube_comments_raw_member_3.csv",
    "/content/youtube_comments_raw_member_4.csv",
]

comments_frames = [pd.read_csv(p, low_memory=False, dtype={"_channel_id":"string","_video_id":"string","_comment_id":"string"}) for p in comments_paths]
comments_team = pd.concat(comments_frames, ignore_index=True, sort=False)

if "_dedupe_key" in comments_team.columns:
    comments_team = comments_team.drop_duplicates(subset=["_dedupe_key"], keep="first").reset_index(drop=True)
else:
    key_cols = [c for c in ["_video_id","_comment_id","comment_text"] if c in comments_team.columns]
    if key_cols:
        comments_team = comments_team.drop_duplicates(subset=key_cols, keep="first").reset_index(drop=True)

comments_team_out = "/content/youtube_team_comments_raw.csv"
comments_team.to_csv(comments_team_out, index=False, encoding="utf-8")

print("Saved:", comments_team_out, comments_team.shape)


videos_paths = [
    "/content/youtube_videos_text_ready_member_1.csv",
    "/content/youtube_videos_text_ready_member_2.csv",
    "/content/youtube_videos_text_ready_member_3.csv",
    "/content/youtube_videos_text_ready_member_4.csv",
]

videos_frames = [pd.read_csv(p, low_memory=False, dtype={"id":"string","_channel_id":"string","_video_id":"string"}) for p in videos_paths]
videos_team = pd.concat(videos_frames, ignore_index=True, sort=False)

# video-level dedupe priority: id -> _video_id -> _dedupe_key
if "id" in videos_team.columns:
    videos_team = videos_team.drop_duplicates(subset=["id"], keep="first").reset_index(drop=True)
elif "_video_id" in videos_team.columns:
    videos_team = videos_team.drop_duplicates(subset=["_video_id"], keep="first").reset_index(drop=True)
elif "_dedupe_key" in videos_team.columns:
    videos_team = videos_team.drop_duplicates(subset=["_dedupe_key"], keep="first").reset_index(drop=True)

videos_team_out = "/content/youtube_team_videos_text_ready.csv"
videos_team.to_csv(videos_team_out, index=False, encoding="utf-8")

print("Saved:", videos_team_out, videos_team.shape)


from google.colab import files

files.download("/content/youtube_team_prd_slim_with_comments.csv")
files.download("/content/youtube_team_comments_raw.csv")
files.download("/content/youtube_team_videos_text_ready.csv")


---

## Phase B — Team Workflow After All Downloads

### B1) Collect files
1. Gather all 4 member ZIP files.
2. Extract all CSV files.
3. You should now have 16 CSV files total.

### B2) Merge by file type (not by member)
1. Merge all `master_full` files into one team file.
2. Merge all `master_slim` files into one team file.
3. Merge all `comments_raw` files into one team file.
4. Merge all `videos_text_ready` files into one team file.

### B3) Deduplicate with correct keys
1. `master_full` and `master_slim`: dedupe by `_dedupe_key`.
2. `comments_raw`: dedupe by `_comment_id`.
3. `videos_text_ready`: dedupe by `id`.

### B4) Save final integrated 4 files
1. `ai_mcn_team_raw_master_all_endpoints_yyyymmdd.csv`
2. `ai_mcn_team_core_master_slim_yyyymmdd.csv`
3. `ai_mcn_team_fact_comments_top10_per_video_yyyymmdd.csv`
4. `ai_mcn_team_fact_videos_with_comment_summary_yyyymmdd.csv`

### B5) Run PRD analyses from integrated files
1. Feature 1 (SNA): mainly `team_fact_videos_with_comment_summary`.
2. Feature 2 (Brand Matching): SNA output + `team_fact_videos_with_comment_summary`.
3. Feature 3 (ROI): `team_fact_videos_with_comment_summary` + manual inputs (budget/CTR/CVR/AOV).
4. Feature 4 (Style + LLM): `team_fact_videos_with_comment_summary` + `team_fact_comments_top10_per_video`.
5. Bias/Ethics: Feature 1 graph outputs (degree vs betweenness/community diversity).

### B6) Frontend build phase
1. Build input fields for brand name, category, target audience, budget, and campaign text.
2. Connect frontend to analysis outputs.
3. Return ranked influencers, score breakdown, ROI range, and strategy suggestions.

### B7) Phase B completion gate
1. Team has 4 integrated CSV files.
2. Team has analysis outputs for Features 1 to 4.
3. Team has a working frontend prototype connected to analysis results.

---
